# Health Disparities in Epilepsy Care

Analysis of geographic, socioeconomic, and racial disparities in epilepsy outcomes using CDC BRFSS data.

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from analyze import generate_state_data, generate_racial_disparity_data, run_regression

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')
np.random.seed(42)

## 1. State-Level Epilepsy Prevalence

Epilepsy affects ~3.4 million Americans (1.2% prevalence). Prevalence varies significantly by state due to demographic, socioeconomic, and healthcare access factors.

In [ ]:
df_state = generate_state_data()
print('State dataset shape:', df_state.shape)
print(df_state.describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Prevalence distribution
axes[0,0].hist(df_state['epilepsy_prevalence_pct'], bins=15, color='#E53935', alpha=0.8, edgecolor='white')
axes[0,0].set_xlabel('Epilepsy Prevalence (%)')
axes[0,0].set_ylabel('Number of States')
axes[0,0].set_title('Distribution of Epilepsy Prevalence Across States')
axes[0,0].axvline(df_state['epilepsy_prevalence_pct'].mean(), color='black', linestyle='--',
                   label=f"Mean: {df_state['epilepsy_prevalence_pct'].mean():.2f}%")
axes[0,0].legend()

# ER visits by Medicaid expansion
expanded = df_state[df_state['medicaid_expanded']==1]['er_visits_per_100k']
not_exp = df_state[df_state['medicaid_expanded']==0]['er_visits_per_100k']
axes[0,1].boxplot([expanded, not_exp], labels=['Medicaid\nExpanded', 'Not\nExpanded'],
                   patch_artist=True,
                   boxprops=dict(facecolor='#42A5F5', alpha=0.7))
t, p = stats.ttest_ind(expanded, not_exp)
axes[0,1].set_ylabel('ER Visits per 100k')
axes[0,1].set_title(f'Seizure ER Visits: Medicaid Expansion\n(t={t:.2f}, p={p:.3f})')
axes[0,1].grid(axis='y', alpha=0.3)

# Scatter: poverty vs ER
colors = ['#42A5F5' if m else '#EF5350' for m in df_state['medicaid_expanded']]
axes[1,0].scatter(df_state['poverty_rate_pct'], df_state['er_visits_per_100k'],
                   c=colors, alpha=0.7, s=60)
m, b, r, p, _ = stats.linregress(df_state['poverty_rate_pct'], df_state['er_visits_per_100k'])
x = np.linspace(df_state['poverty_rate_pct'].min(), df_state['poverty_rate_pct'].max(), 100)
axes[1,0].plot(x, m*x+b, 'k--', alpha=0.6, label=f'r={r:.2f}, p={p:.3f}')
axes[1,0].set_xlabel('Poverty Rate (%)')
axes[1,0].set_ylabel('ER Visits per 100k')
axes[1,0].set_title('Poverty Rate vs. Seizure ER Utilization')
axes[1,0].legend(fontsize=9)

# Neurologist density
axes[1,1].scatter(df_state['neurologists_per_100k'], df_state['er_visits_per_100k'],
                   c='#8E24AA', alpha=0.7, s=60)
m2, b2, r2, p2, _ = stats.linregress(df_state['neurologists_per_100k'], df_state['er_visits_per_100k'])
x2 = np.linspace(df_state['neurologists_per_100k'].min(), df_state['neurologists_per_100k'].max(), 100)
axes[1,1].plot(x2, m2*x2+b2, 'k--', alpha=0.6, label=f'r={r2:.2f}')
axes[1,1].set_xlabel('Neurologists per 100k')
axes[1,1].set_ylabel('ER Visits per 100k')
axes[1,1].set_title('Neurologist Access vs. Seizure ER Utilization')
axes[1,1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../figures/state_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

## 2. Racial Disparities in Epilepsy Management

In [ ]:
df_race = generate_racial_disparity_data()

er_by_race = df_race.groupby('race_ethnicity')['er_visit_year'].agg(['mean','sem']).sort_values('mean', ascending=False)
ins_by_race = df_race.groupby('race_ethnicity')['insured'].mean().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.RdYlBu_r(np.linspace(0.15, 0.85, len(er_by_race)))
axes[0].bar(range(len(er_by_race)), er_by_race['mean'].values, 
             yerr=1.96*er_by_race['sem'].values, color=colors, alpha=0.85, capsize=4)
axes[0].set_xticks(range(len(er_by_race)))
axes[0].set_xticklabels([r.replace(' ', '\n') for r in er_by_race.index], fontsize=8)
axes[0].set_ylabel('Proportion with Seizure ER Visit (annual)')
axes[0].set_title('Racial Disparities in Seizure-Related ER Utilization\n(95% CI shown)', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

for i, (race, row) in enumerate(er_by_race.iterrows()):
    axes[0].text(i, row['mean'] + 0.015, f"{row['mean']:.2f}", ha='center', fontsize=9, fontweight='bold')

coverage = ins_by_race.values * 100
bar_colors = ['#4CAF50' if v > 85 else '#FF9800' if v > 75 else '#F44336' for v in coverage]
axes[1].barh(range(len(ins_by_race)), coverage, color=bar_colors, alpha=0.85)
axes[1].axvline(90, color='black', linestyle='--', alpha=0.5, label='90% target')
axes[1].set_yticks(range(len(ins_by_race)))
axes[1].set_yticklabels([r.replace(' ', '\n') for r in ins_by_race.index], fontsize=8)
axes[1].set_xlabel('Insurance Coverage Rate (%)')
axes[1].set_title('Insurance Coverage by Race/Ethnicity\n(Epilepsy Population)', fontweight='bold')
axes[1].legend()
axes[1].set_xlim(0, 100)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/racial_disparities.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Multivariate Regression: Predictors of ER Utilization

In [ ]:
coefs, r2 = run_regression(df_state)
print(f'Model R² = {r2:.3f}\n')
print('Standardized regression coefficients:')
for k, v in sorted(coefs.items(), key=lambda x: abs(x[1]), reverse=True):
    direction = '+' if v > 0 else '-'
    print(f'  {k:<28} {v:+.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
features = list(coefs.keys())
values = [coefs[f] for f in features]
colors = ['#EF5350' if v > 0 else '#42A5F5' for v in values]
bars = ax.barh(features, values, color=colors, alpha=0.85)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Standardized Coefficient (Effect on ER Visit Rate)')
ax.set_title(f'Predictors of Seizure-Related ER Utilization\n(Linear Regression, R²={r2:.3f})', fontweight='bold')
for bar, v in zip(bars, values):
    ax.text(v + (0.02 if v > 0 else -0.02), bar.get_y() + bar.get_height()/2,
            f'{v:+.3f}', va='center', ha='left' if v > 0 else 'right', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/regression_coefficients.png', bbox_inches='tight', dpi=150)
plt.show()
print('\nKey finding: Uninsured rate and lack of Medicaid expansion are the strongest predictors of high ER utilization.')